# H-A-Alpha Wishart evaluation

In [ ]:
%load_ext autoreload
%autoreload 2

import os
# avoid thread conflicts between numpy and dask
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from polsarpro.dev.devtools import parse_psp_parameter_string
import numpy as np
from pathlib import Path
import xarray as xr
from polsarpro.io import open_netcdf_beam
from polsarpro.dev.io import read_psp_bin
from polsarpro.decompositions import h_a_alpha
from polsarpro.classification import wishart_h_a_alpha
# from polsarpro.classification_dev import wishart_h_a_alpha
from polsarpro.util import plot_h_alpha_plane
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
import shutil

# optional import for progress bar
from dask.diagnostics import ProgressBar

# change to your local C-PolSARpro install dir
c_psp_dir = "/home/c_psp/Soft/bin/"
os.environ["PATH"]+=os.pathsep+f"{c_psp_dir}/data_process_sngl/"
os.environ["PATH"]+=os.pathsep+f"{c_psp_dir}/data_convert/"

# change to your data paths
# original dataset
input_alos_data = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")

# input files from C
input_test_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1/")

# output files from C
output_test_dir = Path("/data/psp/res/wishart_c")
if not os.path.isdir(output_test_dir):
    os.mkdir(output_test_dir)

## Run the C-version on some test data
### Compute H-A-alpha

In [ ]:
# only activate flags for H, A & Alpha

input_dict = dict(
    id=input_test_dir,
    od=output_test_dir,
    iodf="S2T3",
    nwr=7,
    nwc=7,
    ofr=0,
    ofc=0,
    fnr=18432,
    fnc=1248,
    fl1=0,
    fl2=0,
    fl3=1,
    fl4=1,
    fl5=1,
    fl6=0,
    fl7=0,
    fl8=0,
    fl9=0,
    errf="/tmp/MemoryAllocError.txt",
    mask=f"{input_test_dir}/mask_valid_pixels.bin",
)

# making input string: -param1 value1 -param2 value2 etc
param_str = " ".join([f"-{k} {input_dict[k]}" for k in input_dict])
cmd = f"h_a_alpha_decomposition.exe {param_str}"
print(cmd)
os.system(cmd)

# CPSP does not create another config file in output dir
shutil.copy(input_test_dir / "config.txt", output_test_dir)

### Compute Wishart using H-A-Alpha results

In [ ]:
input_dict = dict(
    id=input_test_dir,
    od=output_test_dir,
    iodf="S2",
    nwr=7,
    nwc=7,
    ofr=0,
    ofc=0,
    fnr=18432,
    fnc=1248,
    hf=f"{output_test_dir}/entropy.bin",
    af=f"{output_test_dir}/anisotropy.bin",
    alf=f"{output_test_dir}/alpha.bin",
    nit="10",
    pct="10",
    bmp=0,
    errf="/tmp/MemoryAllocError.txt",
    mask=f"{input_test_dir}/mask_valid_pixels.bin",
)

# making input string: -param1 value1 -param2 value2 etc
param_str = " ".join([f"-{k} {input_dict[k]}" for k in input_dict])
cmd = f"wishart_h_a_alpha_classifier.exe {param_str}"
print(cmd)
os.system(cmd)

## Load ALOS data and C outputs

In [ ]:
# uncomment to test on S matrix made with SNAP
S = open_netcdf_beam(input_alos_data)

out_names = ("entropy", "anisotropy", "alpha")

## Apply the decomposition

In [ ]:
file_out = "/data/psp/res/test_wishart_h_a_alpha.nc"
# netcdf writer cannot overwrite
if os.path.isfile(file_out):
    os.remove(file_out)

with ProgressBar():

    # haa = h_a_alpha(S, boxcar_size=[7, 7], flags=out_names)
    # wishart_h_a_alpha(
    #     S, boxcar_size=[7, 7], h_a_alpha_result=haa, max_iter=10, stop_threshold=10
    # ).to_netcdf(file_out)
    # res = 
    wishart_h_a_alpha(
        S, boxcar_size=[7, 7], max_iter=10, tol_pct=10
    ).to_netcdf(file_out)

# res

# Numerical evaluation

In [ ]:
# open python output (comment if using compute)
out_py = xr.open_dataset("/data/psp/res/test_h_a_alpha.nc")
# open C-PolSARPro outputs
out_c = {}
for name in out_names:
    file_name = output_test_dir / f"{name}.bin"
    out_c[name] = read_psp_bin(file_name)

df = summarize_metrics(out_py, out_c, short_titles=False, verbose=False)
df

In [ ]:
visualize_errors(out_py=out_py, out_c=out_c, clip=False)

In [ ]:
# uncomment to test H/Alpha plane histogram
ax, fig = plot_h_alpha_plane(out_py, bins=500, min_pts=2)
# fig.savefig(output_test_dir / "h_a_alpha_plane.png")